## Instalaciones

In [1]:
!pip install zarr s3fs torch open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 101.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00


## Importaciones

In [2]:
import torch
print(torch.cuda.is_available())      # True si tienes GPU NVIDIA
print(torch.cuda.get_device_name(0))  # nombre de tu GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando:", device)


True
Tesla T4
Usando: cuda


In [ ]:
import math
import random
import os
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
import s3fs
import zarr
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer
import open_clip
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict

## Bloque de la Rama Visual

In [ ]:
class S2Normalizer(nn.Module):
    """
    Normalización z-score por banda para tiles de Sentinel-2.
    Stats calculados sobre el dataset de Cali.
    """
    def __init__(self):
        super().__init__()
        mean = [1931.68, 3341.38, 4165.15, 1960.77]
        std  = [1462.26, 1425.28, 2059.46, 1589.12]
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32).view(1, 4, 1, 1))
        self.register_buffer("std",  torch.tensor(std,  dtype=torch.float32).view(1, 4, 1, 1))
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return (x.float() - self.mean) / (self.std + 1e-6)
 
 
class RemoteCLIP_Visual(nn.Module):
    """
    ViT-B/32 (RemoteCLIP/OpenAI) para Sentinel-2.
 
    INPUT : (B, 4, 64, 64)  — 4 bandas S2 en valores crudos
    OUTPUT: (B, 50, 768)    — [CLS + 49 patches] × 768 dims
                               (ViT-B/32: patches 32×32 → grilla 7×7 = 49)
 
    Pipeline:
        x → S2Normalizer (4 bandas) → band_projection (4→3) →
        interpolate (64→224) → ViT-B/32 sin pooling → (B, 50, 768)
    """
    def __init__(self):
        super().__init__()
        # Normalización z-score
        self.s2_norm = S2Normalizer()
 
        # Proyección aprendible 4→3 bandas (mezcla lineal aprendida)
        # El ViT espera 3 canales; en lugar de descartar una banda,
        # aprendemos qué combinación es más informativa para contaminación
        self.band_projection = nn.Conv2d(4, 3, kernel_size=1, bias=False)
 
        # Cargar Backbone ViT-B/32 preentrenado
        model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
        self.clip_vision = model.visual
        
        # ── NUEVA LÓGICA DE FINE-TUNING PARCIAL (A PRUEBA DE SOBREAJUSTE) ──
        # 1. Congelar inicialmente todo el codificador visual de RemoteCLIP
        for param in self.clip_vision.parameters():
            param.requires_grad = False
            
        # 2. Descongelar únicamente los últimos 2 bloques residuales del Transformer (Bloques 10 y 11)
        num_blocks = len(self.clip_vision.transformer.resblocks)
        for i in range(num_blocks - 2, num_blocks):
            for param in self.clip_vision.transformer.resblocks[i].parameters():
                param.requires_grad = True
                
        # 3. Descongelar el LayerNorm final posterior al transformer
        for param in self.clip_vision.ln_post.parameters():
            param.requires_grad = True
 
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 1. Normalizar las 4 bandas
        x = self.s2_norm(x)  
 
        # 2. Proyectar 4 → 3 canales (aprendible)
        x = self.band_projection(x)         
 
        # 3. Interpolar al tamaño que espera el ViT-B/32
        x = F.interpolate(x, size=(224, 224), mode='bicubic', align_corners=False)
 
        # 4. Forward del ViT sin pooling final (extraemos tokens)
        x = self.clip_vision.conv1(x)                          # (B, 768, 7, 7)
        x = x.reshape(x.shape[0], x.shape[1], -1)             # (B, 768, 49)
        x = x.permute(0, 2, 1)                                 # (B, 49, 768)
 
        cls = (self.clip_vision.class_embedding.to(x.dtype)
               + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device))
        x = torch.cat([cls, x], dim=1)                  
        x = x + self.clip_vision.positional_embedding.to(x.dtype)
        x = self.clip_vision.ln_pre(x)
 
        x = x.permute(1, 0, 2)                              
        x = self.clip_vision.transformer(x)
        x = x.permute(1, 0, 2)                                # (B, 50, 768)
 
        return x 
 
 
class S5PMLP(nn.Module):
    """
    MLP encoder para Sentinel-5P optimizado para 6 canales (Gas + Máscara).
    """
    def __init__(self, d_hidden: int = 128, d_out: int = 256):
        super().__init__()
        # Registramos buffers de normalización para los 3 gases (índices 0, 2 y 4)
        self.register_buffer("mean", torch.tensor([2.5e-4, 1.8e-5, 0.13], dtype=torch.float32))
        self.register_buffer("std",  torch.tensor([1.2e-4, 9.0e-6, 0.02], dtype=torch.float32))
 
        # Cada MLP recibe 2 entradas: (valor_gas, mascara_confianza)
        self.mlp_no2 = self._build_mlp(d_hidden, d_out)
        self.mlp_so2 = self._build_mlp(d_hidden, d_out)
        self.mlp_o3  = self._build_mlp(d_hidden, d_out)
 
        self.pos_embedding = nn.Parameter(torch.randn(3, d_out) * 0.02)
        self.norm = nn.LayerNorm(d_out)
 
    @staticmethod
    def _build_mlp(d_hidden: int, d_out: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Linear(2, d_hidden),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_hidden, d_hidden * 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_hidden * 2, d_out),
        )
 
    def forward(self, s5p_raw: torch.Tensor) -> torch.Tensor:
        # s5p_raw: (B, 6)
        
        # Normalizar los gases (columnas 0, 2, 4)
        no2_norm = (s5p_raw[:, 0] - self.mean[0]) / (self.std[0] + 1e-8)
        so2_norm = (s5p_raw[:, 2] - self.mean[1]) / (self.std[1] + 1e-8)
        o3_norm  = (s5p_raw[:, 4] - self.mean[2]) / (self.std[2] + 1e-8)
        
        # Concatenar con sus respectivas máscaras (columnas 1, 3, 5)
        feat_no2 = torch.stack([no2_norm, s5p_raw[:, 1]], dim=-1)
        feat_so2 = torch.stack([so2_norm, s5p_raw[:, 3]], dim=-1)
        feat_o3  = torch.stack([o3_norm,  s5p_raw[:, 5]], dim=-1)
 
        t_no2 = self.mlp_no2(feat_no2)    # (B, d_out)
        t_so2 = self.mlp_so2(feat_so2)    # (B, d_out)
        t_o3  = self.mlp_o3 (feat_o3)     # (B, d_out)
 
        tokens = torch.stack([t_no2, t_so2, t_o3], dim=1)     # (B, 3, 256)
        tokens = tokens + self.pos_embedding.unsqueeze(0)       
        return self.norm(tokens)
 
 
class CrossAttentionFusion(nn.Module):
    """
    Cross-Attention: tokens S2 (Query) ← tokens S5P (Key/Value)
    Diagrama: Cross-Attention (S2—S5P) → tokens visuales atienden
              tokens espectrales → e_img_raw (512)
 
    Query  : tokens ViT   (B, 50, 768) → proyectados a (B, 50, 256)
    Key    : tokens S5P   (B, 3,  256)
    Value  : tokens S5P   (B, 3,  256)
 
    Salida : e_img_raw (B, 512) + attn_weights (B, 50, 3)
    """
    def __init__(self, d_vit: int = 768, d_s5p: int = 256, d_raw: int = 512):
        super().__init__()
 
        # Proyección Q: 768 → 256 (mismo espacio que K/V)
        self.proj_vit = nn.Linear(d_vit, d_s5p)
 
        # Cross-Attention (embed_dim = d_s5p = 256, 4 cabezas → 64 dims/cabeza)
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_s5p, num_heads=4,
            dropout=0.1, batch_first=True
        )
        self.norm = nn.LayerNorm(d_s5p)
 
        self.ffn = nn.Sequential(
            nn.Linear(d_s5p, d_s5p * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_s5p * 4, d_s5p),
        )
        self.norm2 = nn.LayerNorm(d_s5p)
 
        # Proyección del token CLS fusionado → e_img_raw (512)
        self.to_e_raw = nn.Sequential(
            nn.Linear(d_s5p, d_raw),
            nn.GELU(),
            nn.LayerNorm(d_raw),
        )
 
    def forward(self, vit_tokens: torch.Tensor, s5p_tokens: torch.Tensor):
        """
        vit_tokens : (B, 50, 768)
        s5p_tokens : (B, 3,  256)
        →  e_img_raw   (B, 512)
           attn_weights (B, 50, 3)  ← interpretabilidad
        """
        # 1. Proyectar Query al espacio de 256 dims
        q = self.proj_vit(vit_tokens)             # (B, 50, 256)
 
        # 2. Cross-Attention: Q=visual, K=V=espectral
        attn_out, attn_weights = self.cross_attn(
            query=q, key=s5p_tokens, value=s5p_tokens,
            need_weights=True, average_attn_weights=True
        )
        # attn_out     : (B, 50, 256)
        # attn_weights : (B, 50, 3)  → cuánto atiende cada patch a cada gas
 
        # 3. Residual + LayerNorm
        x = self.norm(q + attn_out)               # (B, 50, 256)
 
        # 4. FFN + residual
        x = self.norm2(x + self.ffn(x))           # (B, 50, 256)
 
        # 5. Token CLS (posición 0) → resumen global del tile fusionado
        cls_fused = x[:, 0, :]                    # (B, 256)
 
        # 6. Proyectar al espacio e_img_raw de 512 dims
        e_img_raw = self.to_e_raw(cls_fused)      # (B, 512)
 
        return e_img_raw, attn_weights

## SAE de la rama visual

In [5]:
class SAEVisual(nn.Module):
    """
    Sparse AutoEncoder para la rama visual.
    Entrada: e_img_raw (B, 512) — salida del CrossAttentionFusion
    
    Encoder : e_img_raw → z_img (B, 512) sparse   [interpretabilidad]
    Decoder : z_img     → x_hat (B, 512)           [reconstrucción]
    Proyect.: e_img_raw → e_img (B, 256) L2-norm   [InfoNCE]
    
    Las neuronas de z_img aprenden a detectar conceptos visuales:
    zonas industriales, vegetación, densidad urbana, etc.
    """
    def __init__(self, d_raw: int = 512, d_out: int = 256, lambda_l1: float = 1e-3):
        super().__init__()
        self.lambda_l1 = lambda_l1

        self.encoder = nn.Sequential(
            nn.Linear(d_raw, d_raw),
            nn.ReLU()
        )
        self.decoder    = nn.Linear(d_raw, d_raw)
        self.projection = nn.Sequential(
            nn.Linear(d_raw, d_out),
            nn.LayerNorm(d_out),
        )

    def forward(self, e_img_raw: torch.Tensor):
        """
        e_img_raw : (B, 512) — fusión imagen+gas del CrossAttention
        Retorna:
            z_img : (B, 512) sparse       → interpretabilidad visual
            x_hat : (B, 512)              → L_recon
            e_img : (B, 256) normalizado  → InfoNCE
        """
        z_img = self.encoder(e_img_raw)                           # (B, 512)
        x_hat = self.decoder(z_img)                               # (B, 512)
        e_img = F.normalize(self.projection(e_img_raw), dim=-1)   # (B, 256)
        return z_img, x_hat, e_img

    def sparsity_ratio(self, z_img: torch.Tensor, threshold: float = 0.01) -> float:
        """KPI: ≥ 0.70. Si es < 0.70 → aumentar lambda_l1."""
        return (z_img.abs() < threshold).float().mean().item()

    def neuron_analysis(self, z_by_class: dict, top_k: int = 10) -> dict:
        """
        Neuronas más activas por clase visual.
        
        Ejemplo de uso en el reporte:
            z_by_class = {
                "contaminacion_alta_NO2": z_industrial,  # (N, 512)
                "vegetacion_densa":       z_verde,        # (N, 512)
            }
            neuronas = sae_visual.neuron_analysis(z_by_class)
            # → {"contaminacion_alta_NO2": [12, 47, 203, ...],
            #    "vegetacion_densa":       [8,  91, 334, ...]}
            # Las neuronas [12, 47, 203] se activan en zonas industriales
            # pero no en zonas verdes → el SAE aprendió a detectar contaminación
        """
        return {
            clase: z.mean(dim=0).topk(top_k).indices.tolist()
            for clase, z in z_by_class.items()
        }

## Bloque de la rama de texto

In [ ]:
class TextEncoder(nn.Module):
    """
    XLM-RoBERTa con ajuste fino parcial de su última capa.
    """
    def __init__(self, model_name: str = 'xlm-roberta-base', d_raw: int = 512):
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        self.proj = nn.Sequential(
            nn.Linear(self.roberta.config.hidden_size, d_raw),
            nn.GELU(),
            nn.LayerNorm(d_raw),
        )
        
        # ── FINE-TUNING SELECCIONADO PARA TEXTO ────────────
        # 1. Congelar todo el backend de RoBERTa por defecto
        for param in self.roberta.parameters():
            param.requires_grad = False
            
        # 2. Descongelar únicamente la última capa del transformer (Capa 11)
        # XLM-RoBERTa-base tiene exactamente 12 capas de transformer (índices 0 a 11)
        num_layers = len(self.roberta.encoder.layer)
        for param in self.roberta.encoder.layer[num_layers - 1].parameters():
            param.requires_grad = True
            
        print(f"-> XLM-RoBERTa inicializado. Capa de transformer {num_layers-1} descongelada para Fine-Tuning.")
 
    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # (B, hidden_size)
        return self.proj(cls)                  # (B, 512)

## SAE de la rama de texto

In [ ]:
class SAETexto(nn.Module):
    """
    Sparse AutoEncoder para la rama textual.
    Entrada: e_txt_raw (B, 512) — salida del TextEncoder (XLM-RoBERTa)

    Encoder : e_txt_raw → z_txt (B, 512) sparse   [interpretabilidad]
    Decoder : z_txt     → x_hat (B, 512)           [reconstrucción]
    Proyect.: e_txt_raw → e_txt (B, 256) L2-norm   [InfoNCE]

    Las neuronas de z_txt aprenden a detectar conceptos semánticos:
    términos de contaminación, ubicaciones geográficas, niveles de gas, etc.
    """
    def __init__(self, d_raw: int = 512, d_out: int = 256, lambda_l1: float = 1e-3):
        super().__init__()
        self.lambda_l1 = lambda_l1

        self.encoder = nn.Sequential(
            nn.Linear(d_raw, d_raw),
            nn.ReLU()
        )
        self.decoder    = nn.Linear(d_raw, d_raw)
        self.projection = nn.Sequential(
            nn.Linear(d_raw, d_out),
            nn.LayerNorm(d_out),
        )

    def forward(self, e_txt_raw: torch.Tensor):
        """
        e_txt_raw : (B, 512) — salida proyectada de XLM-RoBERTa
        Retorna:
            z_txt : (B, 512) sparse       → interpretabilidad textual
            x_hat : (B, 512)              → L_recon
            e_txt : (B, 256) normalizado  → InfoNCE
        """
        z_txt = self.encoder(e_txt_raw)                           # (B, 512)
        x_hat = self.decoder(z_txt)                               # (B, 512)
        e_txt = F.normalize(self.projection(e_txt_raw), dim=-1)   # (B, 256)
        return z_txt, x_hat, e_txt

    def sparsity_ratio(self, z_txt: torch.Tensor, threshold: float = 0.01) -> float:
        return (z_txt.abs() < threshold).float().mean().item()

    def neuron_analysis(self, z_by_class: dict, top_k: int = 10) -> dict:
        """
        Neuronas más activas por clase textual.
        
        Comparar con SAEVisual.neuron_analysis() para encontrar
        neuronas cross-modal: conceptos que el modelo aprendió
        a representar igual en imagen y en texto.
        """
        return {
            clase: z.mean(dim=0).topk(top_k).indices.tolist()
            for clase, z in z_by_class.items()
        }

## Modelo completo

In [8]:
class GeoVisionCLIP(nn.Module):
    """
    Modelo completo — Situación 2 del proyecto.
 
    Rama visual:
        images → ViT → vit_tokens (B,50,768)
        s5p    → S5PMLP → s5p_tokens (B,3,256)
        CrossAttention(vit_tokens, s5p_tokens) → e_img_raw (B,512)
        SAEVisual(e_img_raw) → z_img, x_hat_img, e_img (B,256)
 
    Rama textual:
        texto → XLM-RoBERTa → e_txt_raw (B,512)
        SAETexto(e_txt_raw)  → z_txt, x_hat_txt, e_txt (B,256)
 
    Loss:
        L_InfoNCE(e_img, e_txt, τ)
        L_sae_img = MSE(x_hat_img, e_img_raw) + λ·||z_img||₁
        L_sae_txt = MSE(x_hat_txt, e_txt_raw) + λ·||z_txt||₁
        L_total   = L_InfoNCE + α·(L_sae_img + L_sae_txt)
    """
    def __init__(self, text_model_name: str = 'xlm-roberta-base'):
        super().__init__()
 
        # Rama visual
        self.vit         = RemoteCLIP_Visual()
        self.s5p_encoder = S5PMLP(d_hidden=128, d_out=256)
        self.fusion      = CrossAttentionFusion(d_vit=768, d_s5p=256, d_raw=512)
        self.sae_visual  = SAEVisual(d_raw=512, d_out=256, lambda_l1=1e-3)

        # Rama textual
        self.text_encoder = TextEncoder(model_name=text_model_name, d_raw=512)
        self.sae_texto    = SAETexto(d_raw=512, d_out=256, lambda_l1=2e-3)
        #                                                  ↑ puedes ajustar
        #                                                    independientemente
        #                                                    si uno no alcanza 0.70

        self.logit_scale = nn.Parameter(torch.ones([]) * math.log(1 / 0.07))
 
    def forward(
        self,
        images:         torch.Tensor,   # (B, 4, 64, 64)
        s5p_raw:        torch.Tensor,   # (B, 3)
        input_ids:      torch.Tensor,   # (B, seq_len)
        attention_mask: torch.Tensor,   # (B, seq_len)
    ) -> dict:
 
        # ── Rama visual ───────────────────────────────────────────────
        vit_tokens  = self.vit(images)                        # (B, 50, 768)
        s5p_tokens  = self.s5p_encoder(s5p_raw)               # (B, 3,  256)
        e_img_raw, attn_weights = self.fusion(vit_tokens, s5p_tokens)  # (B, 512)
        z_img, x_hat_img, e_img = self.sae_visual(e_img_raw)  # (B,512),(B,512),(B,256)
 
        # ── Rama textual ──────────────────────────────────────────────
        e_txt_raw = self.text_encoder(input_ids, attention_mask)       # (B, 512)
        z_txt, x_hat_txt, e_txt = self.sae_texto(e_txt_raw)            # (B,512),(B,512),(B,256)
 
        return {
            # Embeddings para InfoNCE
            "e_img":        e_img,          # (B, 256) ← salida final visual
            "e_txt":        e_txt,          # (B, 256) ← salida final textual
            # Entradas al SAE (para L_recon)
            "e_img_raw":    e_img_raw,      # (B, 512)
            "e_txt_raw":    e_txt_raw,      # (B, 512)
            # Reconstrucciones (para L_recon)
            "x_hat_img":    x_hat_img,      # (B, 512)
            "x_hat_txt":    x_hat_txt,      # (B, 512)
            # Representaciones sparse (para interpretabilidad)
            "z_img":        z_img,          # (B, 512)
            "z_txt":        z_txt,          # (B, 512)
            # Temperatura y mapas de atención
            "logit_scale":  self.logit_scale.exp(),
            "attn_weights": attn_weights,   # (B, 50, 3) ← interpretabilidad por gas
        }

## Perdidas del modelo

In [9]:
class GeoVisionLoss(nn.Module):
    def __init__(self, alpha: float = 0.1, lambda_l1: float = 1e-3):
        super().__init__()   # ← esto faltaba
        self.alpha     = alpha
        self.lambda_l1 = lambda_l1

    def info_nce_loss(self, e_img, e_txt, logit_scale):
        logits = logit_scale * e_img @ e_txt.t()
        labels = torch.arange(e_img.size(0), device=e_img.device)
        loss_i = F.cross_entropy(logits,    labels)
        loss_t = F.cross_entropy(logits.t(), labels)
        return (loss_i + loss_t) / 2.0

    def sae_loss(self, x, x_hat, z):
        mse = F.mse_loss(x_hat, x)
        l1  = self.lambda_l1 * z.abs().mean()
        return mse + l1

    def forward(self, outputs: dict):
        l_infonce = self.info_nce_loss(
            outputs["e_img"], outputs["e_txt"], outputs["logit_scale"]
        )
        l_sae_img = self.sae_loss(
            outputs["e_img_raw"], outputs["x_hat_img"], outputs["z_img"]
        )
        l_sae_txt = self.sae_loss(
            outputs["e_txt_raw"], outputs["x_hat_txt"], outputs["z_txt"]
        )
        l_total = l_infonce + self.alpha * (l_sae_img + l_sae_txt)

        return l_total, {
            "loss_total":   l_total.item(),
            "loss_infonce": l_infonce.item(),
            "loss_sae_img": l_sae_img.item(),
            "loss_sae_txt": l_sae_txt.item(),
            "loss_sae":     (l_sae_img + l_sae_txt).item(),
        }

## Descripciones en español

In [10]:
PLANTILLAS = {
    "contaminacion_alta_NO2": [
        "Zona industrial de Yumbo en el norte de Cali con concentración muy alta de dióxido de nitrógeno ({no2:.2e} mol/m²). SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Sector de alta emisión de NO2 ({no2:.2e}) en el corredor industrial de Acopi. SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Imagen satelital del norte de la ciudad con NO2 elevado ({no2:.2e} mol/m²), indicando alto flujo vehicular e industrial. SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Registro crítico de NO2 ({no2:.2e}) cerca de las principales vías de transporte en Cali. SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Niveles de NO2 por encima del percentil 90 ({no2:.2e} mol/m²) en la zona norte periurbana de Cali. SO2: {so2:.2e}, O3: {o3:.2e}."
    ],
    "contaminacion_alta_SO2": [
        "Concentración crítica de SO2 ({so2:.2e} mol/m²) sobre el sector industrial del Valle del Cauca. NO2: {no2:.2e}, O3: {o3:.2e}.",
        "Zona de combustión pesada con SO2 anómalo ({so2:.2e} mol/m²) detectado en el corredor de Yumbo. NO2: {no2:.2e}, O3: {o3:.2e}.",
        "Emisión elevada de dióxido de azufre registrada en la periferia industrial de Cali: {so2:.2e}. NO2: {no2:.2e}, O3: {o3:.2e}.",
        "Fuerte presencia de SO2 ({so2:.2e} mol/m²) indicando quema de combustibles fósiles en el norte de la huella urbana. NO2: {no2:.2e}, O3: {o3:.2e}."
    ],
    "ozono_anomalo": [
        "Concentración anómala de ozono fotoquímico ({o3:.2e} mol/m²) sobre el centro de Cali. NO2: {no2:.2e}, SO2: {so2:.2e}.",
        "Alerta de O3 elevado ({o3:.2e} mol/m²) en el sur de Cali, indicando alta radiación y precursores. NO2: {no2:.2e}, SO2: {so2:.2e}.",
        "Zona con ozono por encima del percentil 90 ({o3:.2e}) afectando las comunas del sur de Santiago de Cali. NO2: {no2:.2e}, SO2: {so2:.2e}.",
        "Episodio de contaminación por O3 ({o3:.2e} mol/m²) en la cuenca atmosférica de la capital del Valle. NO2: {no2:.2e}, SO2: {so2:.2e}."
    ],
    "vegetacion_densa": [
        "Ladera occidental de Cali, cerca a los Farallones, con vegetación forestal densa. Gases estables: NO2: {no2:.2e}, SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Reserva natural de Pance en el sur de Cali con bajo índice de contaminantes. Lecturas: NO2: {no2:.2e}, SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Cobertura verde boscosa en el sector de Cristo Rey con condiciones atmosféricas óptimas. NO2: {no2:.2e}, SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Zona rural andina colindante con la ciudad de Cali exhibiendo aire limpio. NO2: {no2:.2e}, SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Parche de bosque denso y saludable en la cuenca del Río Cali. Gases normales: NO2: {no2:.2e}, SO2: {so2:.2e}, O3: {o3:.2e}."
    ],
    "suelo_urbano": [
        "Área urbana residencial densa en el oriente de Cali (Distrito de Aguablanca). Gases: NO2: {no2:.2e}, SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Tejido urbano altamente edificado en el centro comercial de Cali. Lecturas: NO2: {no2:.2e}, SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Zona residencial consolidada del norte de Cali (barrio Granada y cercanías). Registros estables: NO2: {no2:.2e}, SO2: {so2:.2e}, O3: {o3:.2e}.",
        "Área urbana del oeste de Cali con densidad de construcción moderada. Parámetros: NO2: {no2:.2e}, SO2: {so2:.2e}, O3: {o3:.2e}."
    ],
}
 
def generar_descripcion(clase: str, val_no2: float, val_so2: float, val_o3: float) -> str:
    """
    Genera descripciones dinámicas con variaciones geográficas y de gases
    para evitar colisiones semánticas dentro del lote de entrenamiento.
    """
    plantillas = PLANTILLAS.get(clase, ["Vista satelital de Cali con lecturas: NO2: {no2:.2e}, SO2: {so2:.2e}, O3: {o3:.2e}."])
    plantilla  = random.choice(plantillas)
    
    # Se envían de manera explícita los tres gases para resolver el KeyError
    return plantilla.format(no2=val_no2, so2=val_so2, o3=val_o3)

## Función para extraer datos

In [ ]:
import os
import gc
import s3fs
import zarr
import random
import numpy as np
import torch
from torch.utils.data import Dataset
from tqdm import tqdm
from transformers import AutoTokenizer

class GeoVisionRAMDataset(Dataset):
    def __init__(
        self,
        wasabi_key:    str,
        wasabi_secret: str,
        endpoint:      str,
        s2_path:       str,
        s5p_path:      str,
        n_muestras:    int = 1000,
        model_name:    str = 'xlm-roberta-base',
    ):
        self.cache_dir = "/kaggle/working/geovision_kaggle_cache"
        os.makedirs(self.cache_dir, exist_ok=True)

        self.s2_file_path = os.path.join(self.cache_dir, "s2_data.npy")
        self.s5p_file_path = os.path.join(self.cache_dir, "s5p_data.npy")

        print("1. Conectando a Wasabi S3 (Metadatos ligeros)...")
        s3 = s3fs.S3FileSystem(
            key=wasabi_key,
            secret=wasabi_secret,
            client_kwargs={'endpoint_url': endpoint}
        )
        s2_zarr = zarr.open(s3fs.S3Map(root=s2_path, s3=s3, check=False), mode='r')
        s5p_zarr = zarr.open(s3fs.S3Map(root=s5p_path, s3=s3, check=False), mode='r')

        print("Cargando tiempos S5P...")
        raw_times = np.array(s5p_zarr['time'])

        if np.issubdtype(raw_times.dtype, np.integer):
            max_val = raw_times.max()
            unidad = 's' if max_val < 1e10 else 'ms' if max_val < 1e13 else 'us' if max_val < 1e16 else 'ns'
            s5p_times = raw_times.astype(f'datetime64[{unidad}]')
        else:
            s5p_times = raw_times.astype('datetime64[ns]')

        s5p_times_d = s5p_times.astype('datetime64[D]')
        s5p_lats = np.array(s5p_zarr['lat'])
        s5p_lons = np.array(s5p_zarr['lon'])

        print("2. Calculando cruce espacial...")
        clases_base = ['vegetacion_densa', 'suelo_urbano']
        todos_los_indices = []
        idx_times, idx_lats, idx_lons = [], [], []

        for cls in clases_base:
            if cls in s2_zarr:
                lats_s2 = np.array(s2_zarr[cls]['lats'])
                lons_s2 = np.array(s2_zarr[cls]['lons'])
                fechas_s2 = np.array(s2_zarr[cls]['fechas'])
                for i in range(len(lats_s2)):
                    lat, lon, fecha = lats_s2[i], lons_s2[i], fechas_s2[i]
                    if isinstance(fecha, (bytes, np.bytes_)):
                        fecha_str = fecha.decode('utf-8').strip()
                    else:
                        fecha_str = str(fecha).strip()
                    fecha_dt = np.datetime64(fecha_str, 'D')

                    idx_t = int(np.abs((s5p_times_d - fecha_dt).astype(np.int64)).argmin())
                    idx_la = int(np.abs(s5p_lats - lat).argmin())
                    idx_lo = int(np.abs(s5p_lons - lon).argmin())
                    idx_times.append(idx_t)
                    idx_lats.append(idx_la)
                    idx_lons.append(idx_lo)
                    todos_los_indices.append((cls, i, idx_t, idx_la, idx_lo))

        # 3. Datos de S5P ligeros (Imputación por media)
        no2_raw = s5p_zarr['NO2'].vindex[idx_times, idx_lats, idx_lons]
        so2_raw = s5p_zarr['SO2'].vindex[idx_times, idx_lats, idx_lons]
        o3_raw = s5p_zarr['O3'].vindex[idx_times, idx_lats, idx_lons]
        no2_mask = s5p_zarr['NO2_mascara'].vindex[idx_times, idx_lats, idx_lons]
        so2_mask = s5p_zarr['SO2_mascara'].vindex[idx_times, idx_lats, idx_lons]
        o3_mask = s5p_zarr['O3_mascara'].vindex[idx_times, idx_lats, idx_lons]

        mean_no2, mean_so2, mean_o3 = 2.5e-4, 1.8e-5, 0.13
        no2_vals = np.where(no2_mask == 0.0, mean_no2, no2_raw)
        so2_vals = np.where(so2_mask == 0.0, mean_so2, so2_raw)
        o3_vals = np.where(o3_mask == 0.0, mean_o3, o3_raw)

        # 4. Clasificación por Ranking
        n_total = len(no2_vals)
        top_size = int(n_total * 0.15)
        top_no2_idxs = set(np.argsort(no2_vals)[-top_size:])
        top_so2_idxs = set(np.argsort(so2_vals)[-top_size:])
        top_o3_idxs = set(np.argsort(o3_vals)[-top_size:])

        etiquetas_str = []
        for i in range(n_total):
            if i in top_no2_idxs:
                eti = "contaminacion_alta_NO2"
            elif i in top_so2_idxs:
                eti = "contaminacion_alta_SO2"
            elif i in top_o3_idxs:
                eti = "ozono_anomalo"
            else:
                eti = todos_los_indices[i][0]
            etiquetas_str.append(eti)

        clases_unicas = sorted(set(etiquetas_str))
        self.clase2idx = {c: i for i, c in enumerate(clases_unicas)}
        self.idx2clase = {i: c for c, i in self.clase2idx.items()}

        # 5. Selección Balanceada
        print(f"5. Seleccionando {n_muestras} muestras balanceadas...")
        n_clases = len(clases_unicas)
        muestras_por_clase = n_muestras // n_clases

        indices_por_clase = {c: [] for c in clases_unicas}
        for i, etiqueta in enumerate(etiquetas_str):
            indices_por_clase[etiqueta].append(i)

        idx_sel = []
        random.seed(42)
        for clase, indices in indices_por_clase.items():
            n_disponibles = len(indices)
            if n_disponibles >= muestras_por_clase:
                seleccionados = random.sample(indices, muestras_por_clase)
            else:
                seleccionados = list(indices)
            idx_sel.extend(seleccionados)

        random.shuffle(idx_sel)
        etiquetas_int = np.array([self.clase2idx[e] for e in etiquetas_str])
        self.etiquetas = etiquetas_int[idx_sel].tolist()

        print(f"6. Procesando georreferenciación y fechas de {len(idx_sel)} parches...")
        coordenadas = []
        fechas_list = [] 

        cache_existente = os.path.exists(self.s2_file_path) and os.path.exists(self.s5p_file_path)
        
        if cache_existente:
            print("   -> CACHÉ DETECTADA. Evitando descarga pesada de parches Sentinel-2.")
            for i, idx in enumerate(idx_sel):
                cls, local_idx, _, _, _ = todos_los_indices[idx]
                
                # Extraer coordenada física
                lat_f = s2_zarr[cls]['lats'][local_idx]
                lon_f = s2_zarr[cls]['lons'][local_idx]
                coordenadas.append([lat_f, lon_f])
                
                # Extraer fecha
                fecha_raw = s2_zarr[cls]['fechas'][local_idx]
                fecha_str = fecha_raw.decode('utf-8').strip() if isinstance(fecha_raw, (bytes, np.bytes_)) else str(fecha_raw).strip()
                fecha_clean = fecha_str.replace("np.bytes_(b'", "").replace("')", "")
                fechas_list.append(fecha_clean)
        else:
            print("   -> No se detectó caché local. Descargando parches S2 desde Wasabi...")
            patches = []
            for i, idx in enumerate(idx_sel):
                cls, local_idx, _, _, _ = todos_los_indices[idx]
                patches.append(s2_zarr[cls]['patches'][local_idx])

                lat_f = s2_zarr[cls]['lats'][local_idx]
                lon_f = s2_zarr[cls]['lons'][local_idx]
                coordenadas.append([lat_f, lon_f])
                
                fecha_raw = s2_zarr[cls]['fechas'][local_idx]
                fecha_str = fecha_raw.decode('utf-8').strip() if isinstance(fecha_raw, (bytes, np.bytes_)) else str(fecha_raw).strip()
                fecha_clean = fecha_str.replace("np.bytes_(b'", "").replace("')", "")
                fechas_list.append(fecha_clean)
                
                if (i + 1) % 100 == 0:
                    print(f"   {i+1}/{len(idx_sel)} parches descargados...")

            s2_array = np.stack(patches, axis=0) / 10000.0
            np.save(self.s2_file_path, s2_array)
            del patches, s2_array
            
        self.coordenadas = np.array(coordenadas, dtype=np.float32)
        self.fechas = np.array(fechas_list)  
        
        if not cache_existente:
            s5p_array = np.stack([
                no2_vals[idx_sel], no2_mask[idx_sel],
                so2_vals[idx_sel], so2_mask[idx_sel],
                o3_vals[idx_sel], o3_mask[idx_sel]
            ], axis=1)
            np.save(self.s5p_file_path, s5p_array)
            del s5p_array

        del s2_zarr, s5p_zarr
        gc.collect()

        no2_sel = no2_vals[idx_sel]
        so2_sel = so2_vals[idx_sel]
        o3_sel  = o3_vals[idx_sel]
        
        print("7. Generando descripciones y tokenizando...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.input_ids = []
        self.attn_masks = []

        for i in tqdm(range(len(idx_sel)), desc="Tokenizando"):
            clase_str = self.idx2clase[self.etiquetas[i]]
            desc = generar_descripcion(clase_str, no2_sel[i], so2_sel[i], o3_sel[i])
            tok = self.tokenizer(
                desc, padding='max_length', truncation=True,
                max_length=64, return_tensors="pt"
            )
            self.input_ids.append(tok["input_ids"].squeeze(0))
            self.attn_masks.append(tok["attention_mask"].squeeze(0))

        del no2_sel, so2_sel, o3_sel
        gc.collect()

        self.s2_mmap = np.load(self.s2_file_path, mmap_mode='r')
        self.s5p_mmap = np.load(self.s5p_file_path, mmap_mode='r')
        print(f"\nDataset listo. Lectura SSD activa. No se descargaron parches S2.")

    def __len__(self):
        return len(self.etiquetas)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.s2_mmap[idx], dtype=torch.float32),
            torch.tensor(self.s5p_mmap[idx], dtype=torch.float32),
            self.input_ids[idx],
            self.attn_masks[idx],
        )

    @property
    def etiquetas_array(self):
        return np.array(self.etiquetas)

## Métricas (Recall@k y Sparsity)

In [ ]:
def calcular_recall_semantico(e_img: torch.Tensor, e_txt: torch.Tensor, clases: list, k: int = 1) -> float:
    """
    Recall Semántico @ K: Cuenta como acierto si el texto recuperado
    dentro de las primeras K opciones pertenece a la misma clase semántica
    que la consulta visual original.
    """
    sim = e_img @ e_txt.T  
    top_k_indices = sim.topk(k, dim=1).indices  
    
    hits = 0
    N = len(e_img)
    for i in range(N):
        clase_query = clases[i]
        clases_recuperadas = [clases[idx] for idx in top_k_indices[i].tolist()]
        
        if clase_query in clases_recuperadas:
            hits += 1
            
    return hits / N

def evaluar_recall(model, loader, dataset_maestro, indices_split, device: str) -> dict:
    """Calcula el Recall Semántico @ 1 y @ 5 sobre el conjunto de validación."""
    model.eval()
    all_e_img, all_e_txt = [], []
    with torch.no_grad():
        for images, s5p, input_ids, att_mask in loader:
            out = model(
                images.to(device), s5p.to(device),
                input_ids.to(device), att_mask.to(device)
            )
            all_e_img.append(out["e_img"])
            all_e_txt.append(out["e_txt"])
            
    e_img = torch.cat(all_e_img)
    e_txt = torch.cat(all_e_txt)
    
    # Recuperar las clases síncronas correspondientes a los índices de validación
    clases_val = []
    for idx_global in indices_split:
        clase_idx = dataset_maestro.etiquetas[idx_global]
        clase_str = dataset_maestro.idx2clase[clase_idx]
        clases_val.append(clase_str)
        
    return {
        "recall@1": calcular_recall_semantico(e_img, e_txt, clases_val, k=1),
        "recall@5": calcular_recall_semantico(e_img, e_txt, clases_val, k=5),
    }

## Entrenamiento en 3 fases

In [ ]:
import copy

# FUNCIÓN DE SOPORTE PARA REGISTRO DE HISTORIAL
def _log_history(h, epoch, fase, tl, vl, il, sl, si, st, r1, r5, mse_img, mse_txt):
    h['epoch'].append(epoch)
    h['fase'].append(fase)
    h['train_loss'].append(tl)
    h['val_loss'].append(vl)
    h['infonce_loss'].append(il)
    h['sae_loss'].append(sl)
    h['sparsity_img'].append(si)
    h['sparsity_txt'].append(st)
    h['recall@1'].append(r1)
    h['recall@5'].append(r5)
    h['mse_val_img'].append(mse_img)  # Reconstrucción pura visual (MSE)
    h['mse_val_txt'].append(mse_txt)  # Reconstrucción pura textual (MSE)


# PIPELINE DE ENTRENAMIENTO COMPLETO GeoVision-CLIP
def entrenar(
    model,
    train_loader,
    val_loader,
    dataset_maestro,
    val_indices,
    device:          str   = 'cuda',
    epochs_fase1:    int   = 10,
    epochs_fase2:    int   = 5,
    epochs_fase3:    int   = 5,
    alpha:           float = 0.1,
    lambda_l1:       float = 1e-3,
):
    model     = model.to(device)
    criterion = GeoVisionLoss(alpha=alpha, lambda_l1=lambda_l1)
 
    history = {
        'epoch': [], 'fase': [],
        'train_loss': [], 'val_loss': [],
        'infonce_loss': [], 'sae_loss': [],
        'sparsity_img': [],   
        'sparsity_txt': [],   
        'loss_sae_img': [],   
        'loss_sae_txt': [],   
        'recall@1': [], 'recall@5': [],
        'mse_val_img': [],   
        'mse_val_txt': [],   
    }
 
    # Control para guardar el mejor modelo en RAM (Checkpointing por Recall@5)
    best_recall = -1.0
    best_model_weights = None
 

    # ─────────────────────────────────────────────────────────────
    # FASE 1: Cross-Attention + S5P MLP (alineación en 512)
    # ─────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("FASE 1: Cross-Attention + S5P MLP (alineación en 512)")
    print("="*60)
 
    for param in model.parameters():
        param.requires_grad = False
    for param in model.s5p_encoder.parameters():
        param.requires_grad = True
    for param in model.fusion.parameters():
        param.requires_grad = True
    for param in model.text_encoder.proj.parameters():
        param.requires_grad = True
    for param in model.vit.band_projection.parameters():
        param.requires_grad = True
 
    # Descongelar las últimas 2 capas del transformer de RemoteCLIP y ln_post
    num_blocks = len(model.vit.clip_vision.transformer.resblocks)
    for i in range(num_blocks - 2, num_blocks):
        for param in model.vit.clip_vision.transformer.resblocks[i].parameters():
            param.requires_grad = True
    for param in model.vit.clip_vision.ln_post.parameters():
        param.requires_grad = True
        
    # Descongelar la última capa de RoBERTa
    num_layers = len(model.text_encoder.roberta.encoder.layer)
    for param in model.text_encoder.roberta.encoder.layer[num_layers - 1].parameters():
        param.requires_grad = True
 
    # Agrupar parámetros para optimizador diferencial
    base_params = []
    pretrained_params = []
    for name, param in model.named_parameters():
        if param.requires_grad:
            if "vit.clip_vision" in name or "text_encoder.roberta" in name:
                pretrained_params.append(param)
            else:
                base_params.append(param)
 
    opt_f1 = torch.optim.AdamW([
        {"params": base_params, "lr": 3e-4},
        {"params": pretrained_params, "lr": 1e-5}
    ], weight_decay=1e-2)
    
    sched_f1 = CosineAnnealingLR(opt_f1, T_max=epochs_fase1, eta_min=1e-6)
    params_f1 = [p for p in model.parameters() if p.requires_grad]
 
    for epoch in range(epochs_fase1):
        model.train()
        t_loss, t_sae, t_sp_img, t_sp_txt = 0, 0, 0, 0
        loop = tqdm(train_loader, desc=f"Fase 1 - Época {epoch+1}/{epochs_fase1}")
 
        for images, s5p, input_ids, att_mask in loop:
            images, s5p       = images.to(device), s5p.to(device)
            input_ids, att_mask = input_ids.to(device), att_mask.to(device)
 
            opt_f1.zero_grad()
            out = model(images, s5p, input_ids, att_mask)
 
            e_img_n = F.normalize(out["e_img_raw"], dim=-1)
            e_txt_n = F.normalize(out["e_txt_raw"], dim=-1)
            loss    = criterion.info_nce_loss(e_img_n, e_txt_n, out["logit_scale"])
 
            loss.backward()
            nn.utils.clip_grad_norm_(params_f1, 1.0)
            opt_f1.step()
 
            with torch.no_grad():
                sp_img = model.sae_visual.sparsity_ratio(out['z_img'])   
                sp_txt = model.sae_texto.sparsity_ratio(out['z_txt'])    
                _, ld  = criterion(out)
 
            t_loss   += loss.item()
            t_sae    += ld['loss_sae']
            t_sp_img += sp_img
            t_sp_txt += sp_txt
            loop.set_postfix(InfoNCE=f"{loss.item():.4f}")
 
        sched_f1.step()
 
        # Validación
        model.eval()
        val_loss = 0
        val_mse_img, val_mse_txt = 0, 0
        with torch.no_grad():
            for images, s5p, input_ids, att_mask in val_loader:
                out = model(images.to(device), s5p.to(device),
                            input_ids.to(device), att_mask.to(device))
                e_img_n = F.normalize(out["e_img_raw"], dim=-1)
                e_txt_n = F.normalize(out["e_txt_raw"], dim=-1)
                val_loss += criterion.info_nce_loss(e_img_n, e_txt_n, out["logit_scale"]).item()
                val_mse_img += F.mse_loss(out["x_hat_img"], out["e_img_raw"]).item()
                val_mse_txt += F.mse_loss(out["x_hat_txt"], out["e_txt_raw"]).item()
 
        n_b = len(train_loader)
        n_v = len(val_loader)
        recall = evaluar_recall(model, val_loader, dataset_maestro, val_indices, device)
        _log_history(history, epoch+1, 'Fase1', t_loss/n_b, val_loss/n_v, 
                     t_loss/n_b, t_sae/n_b, t_sp_img/n_b, t_sp_txt/n_b,
                     recall['recall@1'], recall['recall@5'],
                     val_mse_img/n_v, val_mse_txt/n_v)
        
        print(f"  Época {epoch+1:02d} | Train: {t_loss/n_b:.4f} | Val: {val_loss/n_v:.4f}"
              f" | R@1: {recall['recall@1']:.3f} | R@5: {recall['recall@5']:.3f}")
 

    # ─────────────────────────────────────────────────────────────
    # FASE 2: SAEVisual + SAETexto
    # ─────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("FASE 2: SAEs (Reconstrucción + L1)")
    print("="*60)
    
    for param in model.parameters():  
        param.requires_grad = False
    
    for param in model.sae_visual.encoder.parameters():
        param.requires_grad = True
    for param in model.sae_visual.decoder.parameters():
        param.requires_grad = True
    for param in model.sae_texto.encoder.parameters():
        param.requires_grad = True
    for param in model.sae_texto.decoder.parameters():
        param.requires_grad = True
    
    params_f2 = [p for p in model.parameters() if p.requires_grad]
    opt_f2    = torch.optim.AdamW(params_f2, lr=1e-3)
    sched_f2  = CosineAnnealingLR(opt_f2, T_max=epochs_fase2, eta_min=1e-5)
    
    for epoch in range(epochs_fase2):
        model.train()
        t_loss_img, t_loss_txt = 0, 0
        t_sp_img,   t_sp_txt   = 0, 0
    
        loop = tqdm(train_loader, desc=f"Fase 2 - Época {epoch+1}/{epochs_fase2}")
    
        for images, s5p, input_ids, att_mask in loop:
            images, s5p         = images.to(device), s5p.to(device)
            input_ids, att_mask = input_ids.to(device), att_mask.to(device)
    
            opt_f2.zero_grad()
            out = model(images, s5p, input_ids, att_mask)
    
            l_sae_img = criterion.sae_loss(out["e_img_raw"], out["x_hat_img"], out["z_img"])
            l_sae_txt = criterion.sae_loss(out["e_txt_raw"], out["x_hat_txt"], out["z_txt"])
            loss = l_sae_img + l_sae_txt
            loss.backward()
    
            nn.utils.clip_grad_norm_(params_f2, 1.0)
            opt_f2.step()
    
            sp_img = model.sae_visual.sparsity_ratio(out['z_img'])
            sp_txt = model.sae_texto.sparsity_ratio(out['z_txt'])
    
            t_loss_img += l_sae_img.item()
            t_loss_txt += l_sae_txt.item()
            t_sp_img   += sp_img
            t_sp_txt   += sp_txt
    
            loop.set_postfix(
                SAE_img=f"{l_sae_img.item():.4f}", SAE_txt=f"{l_sae_txt.item():.4f}",
                Sp_img=f"{sp_img*100:.1f}%", Sp_txt=f"{sp_txt*100:.1f}%",
            )
    
        sched_f2.step()
        
        # Validación de la Fase 2 (Extracción de MSE Puro)
        model.eval()
        val_loss = 0
        val_mse_img, val_mse_txt = 0, 0
        start_idx = 0
        with torch.no_grad():
            for images, s5p, input_ids, att_mask in val_loader:
                images, s5p = images.to(device), s5p.to(device)
                input_ids, att_mask = input_ids.to(device), att_mask.to(device)
                out = model(images, s5p, input_ids, att_mask)
                
                # Reconstrucción pura de validación (MSE)
                val_mse_img += F.mse_loss(out["x_hat_img"], out["e_img_raw"]).item()
                val_mse_txt += F.mse_loss(out["x_hat_txt"], out["e_txt_raw"]).item()
                
                v_sae_img = criterion.sae_loss(out["e_img_raw"], out["x_hat_img"], out["z_img"])
                v_sae_txt = criterion.sae_loss(out["e_txt_raw"], out["x_hat_txt"], out["z_txt"])
                val_loss += (v_sae_img + v_sae_txt).item()
                
        n_b = len(train_loader)
        n_v = len(val_loader)
        recall = evaluar_recall(model, val_loader, dataset_maestro, val_indices, device)
        _log_history(history, epochs_fase1+epoch+1, 'Fase2', (t_loss_img+t_loss_txt)/n_b, val_loss/n_v,
                     0.0, (t_loss_img+t_loss_txt)/n_b, t_sp_img/n_b, t_sp_txt/n_b,
                     recall['recall@1'], recall['recall@5'],
                     val_mse_img/n_v, val_mse_txt/n_v)
        
        print(f"  Época {epoch+1:02d} | MSE Val Vis: {val_mse_img/n_v:.5f} | MSE Val Txt: {val_mse_txt/n_v:.5f}"
              f" | Sp_Vis: {t_sp_img/n_b*100:.1f}% | Sp_Txt: {t_sp_txt/n_b*100:.1f}%")
 

    # ─────────────────────────────────────────────────────────────
    # FASE 3: Projection heads + logit_scale
    # ─────────────────────────────────────────────────────────────
    print("\n" + "="*60)
    print("FASE 3: Projection heads + τ (espacio InfoNCE 256)")
    print("="*60)
 
    for param in model.parameters():
        param.requires_grad = False
        
    for param in model.sae_visual.projection.parameters():
        param.requires_grad = True
    for param in model.sae_texto.projection.parameters():
        param.requires_grad = True
        
    model.logit_scale.requires_grad = True
 
    params_f3 = [p for p in model.parameters() if p.requires_grad]
    params_f3 += [model.logit_scale]
    opt_f3    = torch.optim.AdamW(params_f3, lr=1e-4)
    sched_f3  = CosineAnnealingLR(opt_f3, T_max=epochs_fase3, eta_min=1e-6)
 
    for epoch in range(epochs_fase3):
        model.train()
        t_loss, t_infonce, t_sae, t_sp_img, t_sp_txt = 0, 0, 0, 0, 0
        loop = tqdm(train_loader, desc=f"Fase 3 - Época {epoch+1}/{epochs_fase3}")
 
        for images, s5p, input_ids, att_mask in loop:
            images, s5p       = images.to(device), s5p.to(device)
            input_ids, att_mask = input_ids.to(device), att_mask.to(device)
 
            opt_f3.zero_grad()
            out           = model(images, s5p, input_ids, att_mask)
            loss, ld      = criterion(out)
 
            loss.backward()
            nn.utils.clip_grad_norm_(params_f3, 1.0)
            opt_f3.step()
 
            sp_img = model.sae_visual.sparsity_ratio(out['z_img'])
            sp_txt = model.sae_texto.sparsity_ratio(out['z_txt'])
            t_loss    += loss.item()
            t_infonce += ld['loss_infonce']
            t_sae     += ld['loss_sae']
            t_sp_img  += sp_img
            t_sp_txt  += sp_txt
            loop.set_postfix(Total=f"{loss.item():.4f}", InfoNCE=f"{ld['loss_infonce']:.4f}")
 
        sched_f3.step()
 
        model.eval()
        val_loss = 0
        val_mse_img, val_mse_txt = 0, 0
        with torch.no_grad():
            for images, s5p, input_ids, att_mask in val_loader:
                out = model(images.to(device), s5p.to(device),
                            input_ids.to(device), att_mask.to(device))
                vl, _ = criterion(out)
                val_loss += vl.item()
                val_mse_img += F.mse_loss(out["x_hat_img"], out["e_img_raw"]).item()
                val_mse_txt += F.mse_loss(out["x_hat_txt"], out["e_txt_raw"]).item()
 
        n_b    = len(train_loader)
        n_v    = len(val_loader)
        recall = evaluar_recall(model, val_loader, dataset_maestro, val_indices, device)
        _log_history(history, epochs_fase1+epochs_fase2+epoch+1, 'Fase3',
                     t_loss/n_b, val_loss/n_v, t_infonce/n_b, t_sae/n_b,
                     t_sp_img/n_b, t_sp_txt/n_b,
                     recall['recall@1'], recall['recall@5'],
                     val_mse_img/n_v, val_mse_txt/n_v)
        
        print(f"  Época {epochs_fase1+epochs_fase2+epoch+1:02d}"
              f" | Total: {t_loss/n_b:.4f} | Val: {val_loss/n_v:.4f}"
              f" | R@1: {recall['recall@1']:.3f} | R@5: {recall['recall@5']:.3f}")
        
        recall_actual = recall['recall@5']
        if recall_actual >= best_recall:
            best_recall = recall_actual
            best_model_weights = copy.deepcopy(model.state_dict())
            print(f"¡Nuevo mejor modelo! Recall@5: {best_recall:.3f} (Weights guardados en RAM)")
 
    # Restaurar el mejor modelo de validación
    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print(f"\nPesos restaurados al MEJOR modelo de validación (Recall@5: {best_recall:.3f})")
        
    # ── REPORTE FINAL DE METRICAS ─────────────────
    print("\n" + "="*60)
    print("REPORTE FINAL DE KPIs ALCANZADOS EN VALIDACIÓN")
    print("="*60)
    
    r1_final = max(history['recall@1'])
    r5_final = max(history['recall@5'])
    sp_visual_final = max(history['sparsity_img'])
    sp_textual_final = max(history['sparsity_txt'])
    mse_visual_final = min(history['mse_val_img'])
    mse_textual_final = min(history['mse_val_txt'])
    
    def eval_status(val, target, is_less=False):
        status = "CUMPLIDO" if (val <= target if is_less else val >= target) else "❌ REPROBADO"
        return status
        
    print(f"1. Recall@1 Imagen->Texto  : {r1_final:.4f}  | Meta: >= 0.45 | {eval_status(r1_final, 0.45)}")
    print(f"2. Recall@5 Imagen->Texto  : {r5_final:.4f}  | Meta: >= 0.70 | {eval_status(r5_final, 0.70)}")
    print(f"3. Sparsity Ratio SAE Vis  : {sp_visual_final*100:.1f}%  | Meta: >= 70%  | {eval_status(sp_visual_final, 0.70)}")
    print(f"4. Sparsity Ratio SAE Txt  : {sp_textual_final*100:.1f}%  | Meta: --     | Informativo")
    print(f"5. Reconstrucción MSE Vis  : {mse_visual_final:.5f}  | Meta: <= 0.05 | {eval_status(mse_visual_final, 0.05, is_less=True)}")
    print(f"6. Reconstrucción MSE Txt  : {mse_textual_final:.5f}  | Meta: --     | Informativo")
    print("="*60)
 
    return history

## Función para las gráficas

In [14]:
def graficar(history, c1: int, c2: int):
    epochs = history['epoch']
    plt.figure(figsize=(16, 12))
 
    kw_v  = dict(color='purple',     linestyle=':', linewidth=2)
    kw_o  = dict(color='darkorange', linestyle=':', linewidth=2)
 
    plt.subplot(3, 2, 1)
    plt.plot(epochs, history['train_loss'], 'o-', label='Train')
    plt.plot(epochs, history['val_loss'],   's--', label='Val')
    plt.axvline(x=c1, **kw_v, label='Fusión→SAE')
    plt.axvline(x=c2, **kw_o, label='SAE→Proj')
    plt.title('Pérdida Total'); plt.xlabel('Época'); plt.legend(); plt.grid(True)
 
    plt.subplot(3, 2, 2)
    plt.plot(epochs, history['infonce_loss'], 'r^-', label='InfoNCE')
    plt.axvline(x=c1, **kw_v); plt.axvline(x=c2, **kw_o)
    plt.title('Pérdida InfoNCE'); plt.xlabel('Época'); plt.legend(); plt.grid(True)
 
    plt.subplot(3, 2, 3)
    plt.plot(epochs, history['sae_loss'], 'gD-', label='SAE (Vis+Txt)')
    plt.axvline(x=c1, **kw_v); plt.axvline(x=c2, **kw_o)
    plt.title('Pérdida SAE'); plt.xlabel('Época'); plt.legend(); plt.grid(True)
 
    plt.subplot(3, 2, 4)
    plt.plot(epochs, [s*100 for s in history['sparsity_img']], 'b.-', label='Visual')
    plt.plot(epochs, [s*100 for s in history['sparsity_txt']], 'm.--', label='Textual')
    plt.axhline(y=70, color='red', linestyle='--', linewidth=1, label='KPI 70%')
    plt.axvline(x=c1, **kw_v); plt.axvline(x=c2, **kw_o)
    plt.title('Sparsity Ratio (KPI ≥ 70%)'); plt.xlabel('Época')
    plt.ylabel('%'); plt.legend(); plt.grid(True)
 
    plt.subplot(3, 2, 5)
    plt.plot(epochs, history['recall@1'], 'co-', label='Recall@1')
    plt.axhline(y=0.45, color='red',    linestyle='--', linewidth=1, label='KPI R@1=0.45')
    plt.axvline(x=c1, **kw_v); plt.axvline(x=c2, **kw_o)
    plt.title('Recall@1 (KPI ≥ 0.45)'); plt.xlabel('Época')
    plt.legend(); plt.grid(True)
 
    plt.subplot(3, 2, 6)
    plt.plot(epochs, history['recall@5'], 'ms-', label='Recall@5')
    plt.axhline(y=0.70, color='red',    linestyle='--', linewidth=1, label='KPI R@5=0.70')
    plt.axvline(x=c1, **kw_v); plt.axvline(x=c2, **kw_o)
    plt.title('Recall@5 (KPI ≥ 0.70)'); plt.xlabel('Época')
    plt.legend(); plt.grid(True)
 
    plt.suptitle("GeoVision-CLIP — Curvas de Entrenamiento (3 Fases)", fontsize=14, weight='bold')
    plt.tight_layout()
    plt.savefig("curvas_geovision.png", dpi=150)
    plt.show()

## Extracción y división de los datos

In [ ]:
# ── Credenciales ─────────────────────────────────────────────
WASABI_KEY      = "TU_KEY"
WASABI_SECRET   = "TU_SECRET"
WASABI_ENDPOINT = "TU_ENDPOINT"
S2_ZARR_PATH    = "TU_PATH"
S5P_ZARR_PATH   = "TU_PATH"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo: {device}")

# ── Dataset ───────────────────────────────────────────────────
dataset = GeoVisionRAMDataset(
    WASABI_KEY, WASABI_SECRET, WASABI_ENDPOINT,
    S2_ZARR_PATH, S5P_ZARR_PATH,
    n_muestras=3500, model_name='xlm-roberta-base'
)


idx     = np.arange(len(dataset))
labels  = dataset.etiquetas_array

# ── VERIFICACIÓN DEL CRUCE TEMPORAL (falla rápido si está roto) ──
n_t = len(set(idx_times))
print(f"   [check] idx_times únicos: {n_t} | combos únicos: {len(set(zip(idx_times, idx_lats, idx_lons)))}")
assert n_t > 1, "Parser de fechas roto: todas las fechas colapsan a 1 instante. Revisa el .decode/.replace."

### División de los datos

In [ ]:
idx_train, idx_temp, _, y_temp = train_test_split(
    idx, labels, train_size=0.70, stratify=labels, random_state=42
)
idx_val, idx_test, _, _ = train_test_split(
    idx_temp, y_temp, train_size=0.50, stratify=y_temp, random_state=42
)
print(f"Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")

## Entrenamiento

In [ ]:
NUEVO_BATCH_SIZE = 8 

train_loader = DataLoader(Subset(dataset, idx_train), batch_size=NUEVO_BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(Subset(dataset, idx_val),   batch_size=NUEVO_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(Subset(dataset, idx_test),  batch_size=NUEVO_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

# ── Modelo ────────────────────────────────────────────────────
modelo = GeoVisionCLIP(text_model_name='xlm-roberta-base').to(device)

# ── Entrenamiento ─────────────────────────────────────────────
EPOCHS_F1, EPOCHS_F2, EPOCHS_F3 = 20, 10, 20

history = entrenar(
    modelo, train_loader, val_loader,
    dataset_maestro=dataset,  # <--- AGREGADO
    val_indices=idx_val,
    device=device,
    epochs_fase1=EPOCHS_F1,
    epochs_fase2=EPOCHS_F2,
    epochs_fase3=EPOCHS_F3,
    alpha=0.1, lambda_l1=2e-3 ,
)

# ── Gráficas ─────────────────────────────────────────────────
graficar(history, c1=EPOCHS_F1, c2=EPOCHS_F1+EPOCHS_F2)

# ── Guardar modelo ────────────────────────────────────────────
torch.save(modelo.state_dict(), "geovision_clip.pt")
print("\nModelo guardado en geovision_clip.pt")

## Evaluación de los resultados

In [ ]:
# EVALUACIÓN EN EL CONJUNTO DE PRUEBA (TEST)
def evaluar_modelo_test(model, test_loader, dataset_maestro, test_indices, device='cuda'):
    """
    Evalúa de forma independiente los KPIs oficiales de la Situación 2
    sobre el conjunto de datos de prueba (Test) no visto durante el entrenamiento.
    """
    print("\n" + "="*60)
    print("EVALUANDO KPIs OFICIALES EN EL CONJUNTO DE PRUEBA (TEST)")
    print("="*60)
    
    model = model.to(device)
    model.eval()
    
    test_mse_visual = 0
    test_mse_textual = 0
    test_sp_visual = 0
    test_sp_textual = 0
    
    all_e_img, all_e_txt = [], []
    
    # 1. Forward pass sobre el Test Loader
    with torch.no_grad():
        for images, s5p, input_ids, att_mask in test_loader:
            images, s5p = images.to(device), s5p.to(device)
            input_ids, att_mask = input_ids.to(device), att_mask.to(device)
            
            out = model(images, s5p, input_ids, att_mask)
            
            # Acumular embeddings para el cálculo global de Recall
            all_e_img.append(out["e_img"])
            all_e_txt.append(out["e_txt"])
            
            # Calcular la reconstrucción pura (MSE) de este lote
            test_mse_visual += F.mse_loss(out["x_hat_img"], out["e_img_raw"]).item()
            test_mse_textual += F.mse_loss(out["x_hat_txt"], out["e_txt_raw"]).item()
            
            # Calcular el ratio de esparcidad (Sparsity) de este lote
            test_sp_visual += model.sae_visual.sparsity_ratio(out["z_img"])
            test_sp_textual += model.sae_texto.sparsity_ratio(out["z_txt"])
            
    # 2. Promediar resultados por lote
    n_batches = len(test_loader)
    avg_mse_vis = test_mse_visual / n_batches
    avg_mse_txt = test_mse_textual / n_batches
    avg_sp_vis = test_sp_visual / n_batches
    avg_sp_txt = test_sp_textual / n_batches
    
    # 3. Concatenar y calcular el Recall Semántico global en Test
    e_img_test = torch.cat(all_e_img, dim=0)
    e_txt_test = torch.cat(all_e_txt, dim=0)
    
    # Recuperar las clases síncronas de test
    clases_test = []
    for idx_global in test_indices:
        clase_idx = dataset_maestro.etiquetas[idx_global]
        clase_str = dataset_maestro.idx2clase[clase_idx]
        clases_test.append(clase_str)
        
    r1_test = calcular_recall_semantico(e_img_test, e_txt_test, clases_test, k=1)
    r5_test = calcular_recall_semantico(e_img_test, e_txt_test, clases_test, k=5)
    
    # 4. Imprimir Reporte de Validación en Test
    def eval_status(val, target, is_less=False):
        return "CUMPLIDO" if (val <= target if is_less else val >= target) else "REPROBADO"
        
    print(f"1. Recall@1 Imagen->Texto  : {r1_test:.4f}  | Meta: >= 0.45 | {eval_status(r1_test, 0.45)}")
    print(f"2. Recall@5 Imagen->Texto  : {r5_test:.4f}  | Meta: >= 0.70 | {eval_status(r5_test, 0.70)}")
    print(f"3. Sparsity Ratio SAE Vis  : {avg_sp_vis*100:.1f}%  | Meta: >= 70%  | {eval_status(avg_sp_vis, 0.70)}")
    print(f"4. Sparsity Ratio SAE Txt  : {avg_sp_txt*100:.1f}%  | Meta: --     | Informativo")
    print(f"5. Reconstrucción MSE Vis  : {avg_mse_vis:.5f}  | Meta: <= 0.05 | {eval_status(avg_mse_vis, 0.05, is_less=True)}")
    print(f"6. Reconstrucción MSE Txt  : {avg_mse_txt:.5f}  | Meta: --     | Informativo")
    print("="*60)
    
    return {
        "r1": r1_test, "r5": r5_test, 
        "sp_vis": avg_sp_vis, "sp_txt": avg_sp_txt, 
        "mse_vis": avg_mse_vis, "mse_txt": avg_mse_txt
    }

# ── Ejecución de la evaluación en Test ─────────────────────────────
reporte_test = evaluar_modelo_test(
    model=modelo,
    test_loader=test_loader,
    dataset_maestro=dataset,
    test_indices=idx_test,
    device=device
)

In [ ]:
# EVALUACIÓN POR CLASE DESPUÉS DEL ENTRENAMIENTO
def evaluar_por_clase(
    model,
    test_loader,
    dataset_maestro,
    test_indices,
    device='cuda',
    topk=(1, 5)
):
    """
    Evalúa desempeño por clase usando retrieval Imagen -> Texto.
    """

    model.eval()
    model = model.to(device)

    all_img_emb = []
    all_txt_emb = []
    clases_test = []

    print("\nExtrayendo embeddings del conjunto TEST...")

    # ─────────────────────────────────────────────
    # Extraer embeddings
    # ─────────────────────────────────────────────
    with torch.no_grad():

        idx_global_actual = 0

        for images, s5p, input_ids, att_mask in test_loader:

            images = images.to(device)
            s5p = s5p.to(device)
            input_ids = input_ids.to(device)
            att_mask = att_mask.to(device)

            out = model(images, s5p, input_ids, att_mask)

            e_img = F.normalize(out["e_img"], dim=-1)
            e_txt = F.normalize(out["e_txt"], dim=-1)

            all_img_emb.append(e_img.cpu())
            all_txt_emb.append(e_txt.cpu())

    # Concatenar
    all_img_emb = torch.cat(all_img_emb, dim=0)
    all_txt_emb = torch.cat(all_txt_emb, dim=0)

    # ─────────────────────────────────────────────
    # Recuperar nombres de clases
    # ─────────────────────────────────────────────
    for idx_global in test_indices:

        clase_idx = dataset_maestro.etiquetas[idx_global]
        clase_str = dataset_maestro.idx2clase[clase_idx]

        clases_test.append(clase_str)

    clases_test = np.array(clases_test)

    # ─────────────────────────────────────────────
    # Matriz de similitud
    # ─────────────────────────────────────────────
    sim_matrix = all_img_emb @ all_txt_emb.T

    # ─────────────────────────────────────────────
    # Métricas por clase
    # ─────────────────────────────────────────────
    resultados = defaultdict(lambda: {
        "total": 0,
        "correct_r1": 0,
        "correct_r5": 0,
        "errores": []
    })

    n = sim_matrix.shape[0]

    for i in range(n):

        clase_real = clases_test[i]

        # Top-K índices recuperados
        ranking = torch.argsort(sim_matrix[i], descending=True)

        top1 = ranking[0].item()
        top5 = ranking[:5].tolist()

        clase_top1 = clases_test[top1]
        clases_top5 = clases_test[top5]

        resultados[clase_real]["total"] += 1

        # Recall@1
        if clase_real == clase_top1:
            resultados[clase_real]["correct_r1"] += 1
        else:
            resultados[clase_real]["errores"].append(clase_top1)

        # Recall@5
        if clase_real in clases_top5:
            resultados[clase_real]["correct_r5"] += 1

    # ─────────────────────────────────────────────
    # Construcción del reporte
    # ─────────────────────────────────────────────
    filas = []

    for clase, datos in resultados.items():

        total = datos["total"]

        r1 = datos["correct_r1"] / total
        r5 = datos["correct_r5"] / total

        # Clase más confundida
        if len(datos["errores"]) > 0:
            conf_mas_comun = pd.Series(datos["errores"]).value_counts().idxmax()
        else:
            conf_mas_comun = "-"

        filas.append({
            "Clase": clase,
            "Samples": total,
            "Recall@1": round(r1, 4),
            "Recall@5": round(r5, 4),
            "Correctos@1": datos["correct_r1"],
            "Correctos@5": datos["correct_r5"],
            "Clase_mas_confundida": conf_mas_comun
        })

    df_resultados = pd.DataFrame(filas)

    # Ordenar por peor Recall@1
    df_resultados = df_resultados.sort_values(
        by="Recall@1",
        ascending=True
    ).reset_index(drop=True)

    # ─────────────────────────────────────────────
    # Resultados generales
    # ─────────────────────────────────────────────
    print("\n" + "="*70)
    print("REPORTE POR CLASE")
    print("="*70)

    print(df_resultados)

    print("\n" + "="*70)
    print("MEJORES CLASES")
    print("="*70)

    print(df_resultados.sort_values(
        by="Recall@1",
        ascending=False
    ).head(10))

    print("\n" + "="*70)
    print("PEORES CLASES")
    print("="*70)

    print(df_resultados.head(10))

    return df_resultados


df_clases = evaluar_por_clase(
    model=modelo,
    test_loader=test_loader,
    dataset_maestro=dataset,
    test_indices=idx_test,
    device=device
)

## Extracción de los embeddings

In [ ]:
def extraer_representaciones_geovision(model, dataloader, dataset_maestro, indices_split, device='cuda'):
    """
    Extrae representaciones latentes, coordenadas y fechas síncronas del dataset.
    """
    model.eval()
    model.to(device)
    
    e_img_list = []
    e_txt_list = []
    z_img_list = []
    z_txt_list = []
    clases_list = []
    gases_list = []
    coordenadas_list = []
    fechas_list = []  # <--- NUEVO
    
    print("Iniciando extracción de variables...")
    
    with torch.no_grad():
        for images, s5p, input_ids, att_mask in dataloader:
            images, s5p = images.to(device), s5p.to(device)
            input_ids, att_mask = input_ids.to(device), att_mask.to(device)
            
            outputs = model(images, s5p, input_ids, att_mask)
            
            e_img_list.append(outputs["e_img"].cpu().numpy())
            e_txt_list.append(outputs["e_txt"].cpu().numpy())
            z_img_list.append(outputs["z_img"].cpu().numpy())
            z_txt_list.append(outputs["z_txt"].cpu().numpy())
            gases_list.append(s5p.cpu().numpy())
            
    # Concatenar arrays
    e_img_array = np.concatenate(e_img_list, axis=0)
    e_txt_array = np.concatenate(e_txt_list, axis=0)
    z_img_array = np.concatenate(z_img_list, axis=0)
    z_txt_array = np.concatenate(z_txt_list, axis=0)
    gases_array = np.concatenate(gases_list, axis=0)
    
    # Extraer etiquetas, coordenadas y fechas síncronas
    for idx_global in indices_split:
        clase_idx = dataset_maestro.etiquetas[idx_global]
        clase_str = dataset_maestro.idx2clase[clase_idx]
        clases_list.append(clase_str)
        
        coord = dataset_maestro.coordenadas[idx_global]
        coordenadas_list.append(coord)
        
        # Capturar la fecha correspondiente
        fecha_str = dataset_maestro.fechas[idx_global]  # <--- NUEVO
        fechas_list.append(fecha_str)
        
    coordenadas_array = np.array(coordenadas_list, dtype=np.float32)
    fechas_array = np.array(fechas_list)  # <--- NUEVO
        
    print("Extracción completada de forma síncrona.")
    return {
        "e_img": e_img_array,
        "e_txt": e_txt_array,
        "z_img": z_img_array,
        "z_txt": z_txt_array,
        "clases": clases_list,
        "gases": gases_array,
        "coordenadas": coordenadas_array,
        "fechas": fechas_array  # <--- NUEVO
    }


# 1. Correr la extracción sobre el conjunto de validación (o entrenamiento)
representaciones_train = extraer_representaciones_geovision(
    model=modelo,
    dataloader=train_loader,
    dataset_maestro=dataset,  
    indices_split=idx_train, 
    device=device
)

# 2. Ruta del archivo donde desea guardarlo
nombre_archivo = "representaciones_geovision.npz"

# 2. Guardar todos los arrays comprimidos en un solo archivo
np.savez_compressed(
    nombre_archivo,
    e_img=representaciones_train["e_img"],
    e_txt=representaciones_train["e_txt"],
    z_img=representaciones_train["z_img"],
    z_txt=representaciones_train["z_txt"],
    clases=np.array(representaciones_train["clases"]),  # Convertimos la lista de strings a array
    gases=representaciones_train["gases"],
    coordenadas=representaciones_train["coordenadas"],
    fechas=representaciones_train["fechas"]
)

print(f"Representaciones guardadas de forma segura en '{nombre_archivo}'")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EVALUACIÓN POR CLASE DESPUÉS DEL ENTRENAMIENTO
# ═══════════════════════════════════════════════════════════════════

import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from collections import defaultdict

def evaluar_por_clase(
    model,
    test_loader,
    dataset_maestro,
    test_indices,
    device='cuda',
    topk=(1, 5)
):
    """
    Evalúa desempeño por clase usando retrieval Imagen -> Texto.
    """

    model.eval()
    model = model.to(device)

    all_img_emb = []
    all_txt_emb = []
    clases_test = []

    print("\nExtrayendo embeddings del conjunto TEST...")

    # ─────────────────────────────────────────────
    # Extraer embeddings
    # ─────────────────────────────────────────────
    with torch.no_grad():

        idx_global_actual = 0

        for images, s5p, input_ids, att_mask in test_loader:

            images = images.to(device)
            s5p = s5p.to(device)
            input_ids = input_ids.to(device)
            att_mask = att_mask.to(device)

            out = model(images, s5p, input_ids, att_mask)

            e_img = F.normalize(out["e_img"], dim=-1)
            e_txt = F.normalize(out["e_txt"], dim=-1)

            all_img_emb.append(e_img.cpu())
            all_txt_emb.append(e_txt.cpu())

    # Concatenar
    all_img_emb = torch.cat(all_img_emb, dim=0)
    all_txt_emb = torch.cat(all_txt_emb, dim=0)

    # ─────────────────────────────────────────────
    # Recuperar nombres de clases
    # ─────────────────────────────────────────────
    for idx_global in test_indices:

        clase_idx = dataset_maestro.etiquetas[idx_global]
        clase_str = dataset_maestro.idx2clase[clase_idx]

        clases_test.append(clase_str)

    clases_test = np.array(clases_test)

    # ─────────────────────────────────────────────
    # Matriz de similitud
    # ─────────────────────────────────────────────
    sim_matrix = all_img_emb @ all_txt_emb.T

    # ─────────────────────────────────────────────
    # Métricas por clase
    # ─────────────────────────────────────────────
    resultados = defaultdict(lambda: {
        "total": 0,
        "correct_r1": 0,
        "correct_r5": 0,
        "errores": []
    })

    n = sim_matrix.shape[0]

    for i in range(n):

        clase_real = clases_test[i]

        # Top-K índices recuperados
        ranking = torch.argsort(sim_matrix[i], descending=True)

        top1 = ranking[0].item()
        top5 = ranking[:5].tolist()

        clase_top1 = clases_test[top1]
        clases_top5 = clases_test[top5]

        resultados[clase_real]["total"] += 1

        # Recall@1
        if clase_real == clase_top1:
            resultados[clase_real]["correct_r1"] += 1
        else:
            resultados[clase_real]["errores"].append(clase_top1)

        # Recall@5
        if clase_real in clases_top5:
            resultados[clase_real]["correct_r5"] += 1

    # ─────────────────────────────────────────────
    # Construcción del reporte
    # ─────────────────────────────────────────────
    filas = []

    for clase, datos in resultados.items():

        total = datos["total"]

        r1 = datos["correct_r1"] / total
        r5 = datos["correct_r5"] / total

        # Clase más confundida
        if len(datos["errores"]) > 0:
            conf_mas_comun = pd.Series(datos["errores"]).value_counts().idxmax()
        else:
            conf_mas_comun = "-"

        filas.append({
            "Clase": clase,
            "Samples": total,
            "Recall@1": round(r1, 4),
            "Recall@5": round(r5, 4),
            "Correctos@1": datos["correct_r1"],
            "Correctos@5": datos["correct_r5"],
            "Clase_mas_confundida": conf_mas_comun
        })

    df_resultados = pd.DataFrame(filas)

    # Ordenar por peor Recall@1
    df_resultados = df_resultados.sort_values(
        by="Recall@1",
        ascending=True
    ).reset_index(drop=True)

    # ─────────────────────────────────────────────
    # Resultados generales
    # ─────────────────────────────────────────────
    print("\n" + "="*70)
    print("REPORTE POR CLASE")
    print("="*70)

    print(df_resultados)

    print("\n" + "="*70)
    print("MEJORES CLASES")
    print("="*70)

    print(df_resultados.sort_values(
        by="Recall@1",
        ascending=False
    ).head(10))

    print("\n" + "="*70)
    print("PEORES CLASES")
    print("="*70)

    print(df_resultados.head(10))

    return df_resultados


# ═══════════════════════════════════════════════════════════════════
# EJECUCIÓN
# ═══════════════════════════════════════════════════════════════════

df_clases = evaluar_por_clase(
    model=modelo,
    test_loader=test_loader,
    dataset_maestro=dataset,
    test_indices=idx_test,
    device=device
)

## Interpretabilidad Mécanica

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. CARGAR EL ARCHIVO NPZ
datos = np.load("representaciones_geovision.npz", allow_pickle=True)

# 2. Extraer las variables necesarias
z_img_val = datos["z_img"]  # Matriz de activaciones latentes [N, 512]
clases_val = datos["clases"] # Lista de etiquetas correspondientes

# 3. Crear el DataFrame con las activaciones latentes y sus respectivas clases
df_sae = pd.DataFrame(z_img_val)
df_sae["clase"] = clases_val  # Ahora 'clases_val' está cargada correctamente

# 4. Calcular la activación promedio de cada una de las 512 neuronas para las 5 clases
activacion_promedio = df_sae.groupby("clase").mean() # Matriz [5 clases, 512 neuronas]

# 5. Identificar las neuronas más representativas de cada clase (Top 3 neuronas por clase)
neuronas_clave = set()
for clase in activacion_promedio.index:
    top_neuronas = activacion_promedio.loc[clase].nlargest(3).index.tolist()
    neuronas_clave.update(top_neuronas)
    
neuronas_clave = sorted(list(neuronas_clave))

# 6. Graficar el mapa de calor con las neuronas más selectivas
plt.figure(figsize=(14, 6))
sns.heatmap(
    activacion_promedio[neuronas_clave], 
    cmap="YlOrRd", 
    cbar_kws={'label': 'Intensidad de Activación Promedio (z)'},
    linewidths=0.5
)
plt.title('Mapa de Calor de Interpretabilidad Mecánica (Selectividad de Neuronas del SAE Visual)', fontsize=13, weight='bold', pad=15)
plt.xlabel('Índice de la Neurona Latente del SAE (z)', fontsize=11)
plt.ylabel('Clase Semántica', fontsize=11)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("interpretabilidad_heatmap.png", dpi=150)
plt.show()

In [ ]:
# 1. Cargar las representaciones
datos = np.load("representaciones_geovision.npz", allow_pickle=True)
z_img = datos["z_img"]  
clases = datos["clases"] 

df_sae = pd.DataFrame(z_img)
df_sae["Clase"] = clases
activacion_promedio = df_sae.groupby("Clase").mean()  # Matriz [5 x 512]

# 2. CALCULAR EL ÍNDICE DE SELECTIVIDAD RELATIVA
# Evitamos divisiones por cero con un épsilon
epsilon = 1e-8
suma_total_activacion = activacion_promedio.sum(axis=0)
matriz_selectividad = activacion_promedio / (suma_total_activacion + epsilon)

# 3. Seleccionar las 3 neuronas con mayor selectividad RELATIVA para cada clase
neuronas_monosemanticas = set()
for clase in matriz_selectividad.index:
    # Tomamos las 3 neuronas con mayor concentración relativa en esta clase
    top_selectivas = matriz_selectividad.loc[clase].nlargest(3).index.tolist()
    neuronas_monosemanticas.update(top_selectivas)

neuronas_monosemanticas = sorted(list(neuronas_monosemanticas))

# 4. Reestructurar los datos aplicando un umbral para conservar SOLO los picos de especialización
datos_burbuja_exclusiva = []
for clase in activacion_promedio.index:
    for neurona in neuronas_monosemanticas:
        activacion = activacion_promedio.loc[clase, neurona]
        
        # FILTRO DE EXCLUSIVIDAD: Solo graficamos si la neurona tiene una activación alta (> 0.11)
        # Esto elimina el ruido amarillo de fondo y deja únicamente las burbujas de especialización pura
        if activacion > 0.11: 
            datos_burbuja_exclusiva.append({
                "Clase Semántica": clase,
                "Neurona Monosemántica (z)": f"N_{neurona:03d}",
                "Intensidad de Activación": activacion
            })

df_burbuja_exclusiva = pd.DataFrame(datos_burbuja_exclusiva)

# 5. Graficar el nuevo Mapa de Burbujas Monosemánticas
plt.figure(figsize=(14, 6))
sns.set_theme(style="whitegrid")

# El tamaño de la burbuja se escala dinámicamente para que siempre sea visible
max_activation = df_burbuja_exclusiva["Intensidad de Activación"].max()
sizes = (df_burbuja_exclusiva["Intensidad de Activación"] / max_activation) * 500

scatter = plt.scatter(
    x=df_burbuja_exclusiva["Neurona Monosemántica (z)"],
    y=df_burbuja_exclusiva["Clase Semántica"],
    s=sizes + 50,  # Evita que las burbujas muy pequeñas tengan tamaño cero
    c=df_burbuja_exclusiva["Intensidad de Activación"],
    cmap="YlOrRd",
    alpha=0.85,
    edgecolors="black",
    linewidths=0.75
)

cbar = plt.colorbar(scatter, pad=0.02)
cbar.set_label('Activación Promedio (z)', fontsize=10, weight='semibold')

plt.title('Interpretabilidad Mecánica: Neuronas Monosemánticas Exclusivas del SAE Visual', fontsize=13, weight='bold', pad=15)
plt.xlabel('Índice de la Neurona Latente Especializada (z)', fontsize=11, labelpad=10)
plt.ylabel('Clase Semántica de Validación', fontsize=11, labelpad=10)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig("interpretabilidad_bubble_plot_monosemantico.png", dpi=150)
plt.show()